# News Summary QLoRA Fine-Tune

Notebook này fine-tune `Qwen/Qwen2.5-3B-Instruct` trên `sunnysai12345/news-summary` bằng `Unsloth + QLoRA`, giữ nguyên target output `JSON summary` như baseline hiện tại. Checkpointing được thiết kế theo kiểu `archive + latest ref + best ref` để resume an toàn trên Colab 1 GPU mà không nhân đôi dung lượng checkpoint.

Trước khi train, hãy chạy `notebooks/model_eval_news_summary_colab.ipynb` để tạo baseline artifacts và đặt `NEWS_SUMMARY_BASELINE_MANIFEST_PATH` nếu muốn so sánh before/after hoặc khóa đúng protocol đánh giá.

Trước khi tải dataset, cung cấp Kaggle credentials bằng `KAGGLE_USERNAME`/`KAGGLE_KEY` hoặc đặt `kaggle.json` tại `/content/drive/MyDrive/.kaggle/kaggle.json` sau khi mount Google Drive.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import inspect
import os
import subprocess
import sys
from datetime import datetime, timezone

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    print(f'Running outside Colab: {exc}')

def find_repo_root(*starts: Path) -> Path | None:
    seen = set()
    for start in starts:
        if not start:
            continue
        for candidate in [start, *start.parents]:
            key = str(candidate.resolve()) if candidate.exists() else str(candidate)
            if key in seen:
                continue
            seen.add(key)
            if (candidate / 'reasoning_nlp').exists():
                return candidate
    return None

DEFAULT_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.cwd()
if IN_COLAB:
    REPO_DIR = Path('/content/video-summary')
    BRANCH_NAME = os.environ.get('VIDEO_SUMMARY_BRANCH', 'main').strip() or 'main'
    if not REPO_DIR.exists():
        subprocess.check_call([
            'git', 'clone', '--single-branch', '--branch', BRANCH_NAME,
            'https://github.com/TCTri205/video-summary.git', str(REPO_DIR)
        ])
    else:
        os.chdir(REPO_DIR)
        subprocess.check_call(['git', 'fetch', 'origin'])
        subprocess.check_call(['git', 'checkout', BRANCH_NAME])
        subprocess.check_call(['git', 'pull', 'origin', BRANCH_NAME])
    os.chdir(REPO_DIR)

SEARCH_STARTS = [
    Path.cwd(),
    Path(__file__).resolve().parent if '__file__' in globals() else None,
    Path('/content/video-summary') if IN_COLAB else None,
    Path('/content/drive/MyDrive/video-summary') if IN_COLAB else None,
]
REPO_ROOT = find_repo_root(*[path for path in SEARCH_STARTS if path is not None])
if REPO_ROOT is None:
    raise FileNotFoundError(
        'Cannot locate repo root containing reasoning_nlp. Clone or open the video-summary repo, '
        'or set the notebook working directory inside that repo before running imports.'
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REQUIRED_PACKAGES = {
    'kaggle': 'kaggle',
    'datasets': 'datasets',
    'trl': 'trl==0.23.1',
    'transformers': 'transformers==4.56.1',
    'accelerate': 'accelerate',
    'bitsandbytes': 'bitsandbytes',
    'peft': 'peft',
    'unsloth': 'unsloth',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'bert_score': 'bert-score',
    'rouge_score': 'rouge-score',
}
missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from datasets import Dataset
from IPython.display import Markdown, display
import pandas as pd
import torch
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

import reasoning_nlp.eval.news_summary_baseline as news_summary_baseline_module
import reasoning_nlp.eval.news_summary_finetune as news_summary_finetune_module
news_summary_baseline_module = importlib.reload(news_summary_baseline_module)
news_summary_finetune_module = importlib.reload(news_summary_finetune_module)
print('news_summary_baseline module =', inspect.getsourcefile(news_summary_baseline_module))
print('news_summary_finetune module =', inspect.getsourcefile(news_summary_finetune_module))

from reasoning_nlp.eval.news_summary_baseline import (
    build_production_adapted_prompt_profile,
    download_dataset_if_needed,
    read_csv_robust,
    resolve_csv_file,
    run_baseline_evaluation,
)
from reasoning_nlp.eval.news_summary_finetune import (
    NewsSummaryFineTuneConfig,
    build_before_after_comparison,
    build_checkpoint_callback,
    resolve_colab_safe_profile,
    resolve_runtime_precision,
    build_sft_config_kwargs,
    build_post_train_baseline_config,
    build_run_paths,
    build_training_manifest,
    load_baseline_manifest,
    load_training_dataframe,
    prepare_training_records,
    refresh_checkpoint_index,
    require_baseline_manifest,
    resolve_frozen_eval_ids_path,
    resolve_resume_checkpoint,
    resolve_training_split_config,
    split_train_eval_by_frozen_ids,
    validate_baseline_protocol_compatibility,
    write_training_manifest,
)

pd.set_option('display.max_colwidth', 120)
print('REPO_ROOT =', REPO_ROOT)
print('DEFAULT_ROOT =', DEFAULT_ROOT)


In [ ]:
BASE_MODEL_NAME = os.environ.get('VIDEO_SUMMARY_LOCAL_MODEL_VERSION', 'Qwen/Qwen2.5-3B-Instruct').strip()
DATASET_SLUG = os.environ.get('KAGGLE_DATASET_SLUG', 'sunnysai12345/news-summary').strip()
CACHE_DIR = Path(os.environ.get('NEWS_SUMMARY_CACHE_DIR', str(DEFAULT_ROOT / 'video-summary-cache' / 'news-summary')))
OUTPUT_ROOT = Path(os.environ.get('NEWS_SUMMARY_FINETUNE_OUTPUT_ROOT', str(DEFAULT_ROOT / 'video-summary-finetune')))
CHECKPOINT_ROOT = Path(os.environ.get('NEWS_SUMMARY_FINETUNE_CHECKPOINT_ROOT', str(OUTPUT_ROOT / 'runs')))
KAGGLE_JSON_DRIVE_PATH = Path(os.environ.get('KAGGLE_JSON_DRIVE_PATH', str(DEFAULT_ROOT / '.kaggle' / 'kaggle.json')))
BASELINE_MANIFEST_PATH_RAW = os.environ.get('NEWS_SUMMARY_BASELINE_MANIFEST_PATH', '').strip()
BASELINE_MANIFEST_PATH = Path(BASELINE_MANIFEST_PATH_RAW) if BASELINE_MANIFEST_PATH_RAW else None
CSV_FILENAME = os.environ.get('NEWS_SUMMARY_CSV_FILENAME', 'news_summary.csv').strip()
ARTICLE_COLUMN = os.environ.get('NEWS_SUMMARY_ARTICLE_COLUMN', 'ctext').strip()
SUMMARY_COLUMN = os.environ.get('NEWS_SUMMARY_SUMMARY_COLUMN', 'text').strip()
AUX_HEADLINE_COLUMN = os.environ.get('NEWS_SUMMARY_HEADLINE_COLUMN', 'headlines').strip()
SPLIT_COLUMN = os.environ.get('NEWS_SUMMARY_SPLIT_COLUMN', '').strip()
TARGET_SPLIT = os.environ.get('NEWS_SUMMARY_TARGET_SPLIT', '').strip()
RUN_NAME = os.environ.get('NEWS_SUMMARY_FINETUNE_RUN_NAME', datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_qwen_news_qlora'))
FORCE_REDOWNLOAD = os.environ.get('NEWS_SUMMARY_FORCE_REDOWNLOAD', 'false').strip().lower() in {'1', 'true', 'yes'}
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
GPU_SAFE_PROFILE = resolve_colab_safe_profile(GPU_NAME)
PRECISION_CONFIG = resolve_runtime_precision(bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)()))
MAX_EVAL_SAMPLES = int(os.environ.get('NEWS_SUMMARY_MAX_EVAL_SAMPLES', str(GPU_SAFE_PROFILE.max_eval_samples)))
RANDOM_SEED = int(os.environ.get('NEWS_SUMMARY_RANDOM_SEED', '42'))
MAX_SEQ_LENGTH = int(os.environ.get('NEWS_SUMMARY_MAX_SEQ_LENGTH', str(GPU_SAFE_PROFILE.max_seq_length)))
LEARNING_RATE = float(os.environ.get('NEWS_SUMMARY_LEARNING_RATE', '2e-4'))
NUM_TRAIN_EPOCHS = float(os.environ.get('NEWS_SUMMARY_NUM_TRAIN_EPOCHS', '2.0'))
PER_DEVICE_TRAIN_BATCH_SIZE = int(os.environ.get('NEWS_SUMMARY_PER_DEVICE_TRAIN_BATCH_SIZE', str(GPU_SAFE_PROFILE.per_device_train_batch_size)))
PER_DEVICE_EVAL_BATCH_SIZE = int(os.environ.get('NEWS_SUMMARY_PER_DEVICE_EVAL_BATCH_SIZE', str(GPU_SAFE_PROFILE.per_device_eval_batch_size)))
GRAD_ACCUM_STEPS = int(os.environ.get('NEWS_SUMMARY_GRAD_ACCUM_STEPS', str(GPU_SAFE_PROFILE.gradient_accumulation_steps)))
WARMUP_STEPS = int(os.environ.get('NEWS_SUMMARY_WARMUP_STEPS', '20'))
SAVE_STEPS = int(os.environ.get('NEWS_SUMMARY_SAVE_STEPS', '50'))
EVAL_STEPS = int(os.environ.get('NEWS_SUMMARY_EVAL_STEPS', '50'))
LOGGING_STEPS = int(os.environ.get('NEWS_SUMMARY_LOGGING_STEPS', '10'))
SAVE_TOTAL_LIMIT = int(os.environ.get('NEWS_SUMMARY_SAVE_TOTAL_LIMIT', '2'))
LORA_RANK = int(os.environ.get('NEWS_SUMMARY_LORA_RANK', str(GPU_SAFE_PROFILE.lora_rank)))
LORA_ALPHA = int(os.environ.get('NEWS_SUMMARY_LORA_ALPHA', str(GPU_SAFE_PROFILE.lora_alpha)))
LORA_DROPOUT = float(os.environ.get('NEWS_SUMMARY_LORA_DROPOUT', '0.0'))
WEIGHT_DECAY = float(os.environ.get('NEWS_SUMMARY_WEIGHT_DECAY', '0.01'))
LOAD_IN_4BIT = os.environ.get('NEWS_SUMMARY_LOAD_IN_4BIT', 'true').strip().lower() not in {'0', 'false', 'no'}
EARLY_STOPPING_PATIENCE = int(os.environ.get('NEWS_SUMMARY_EARLY_STOPPING_PATIENCE', '2'))
ENABLE_BASELINE_COMPARISON_RAW = os.environ.get('NEWS_SUMMARY_ENABLE_BASELINE_COMPARISON', 'auto').strip().lower()
if ENABLE_BASELINE_COMPARISON_RAW in {'', 'auto'}:
    ENABLE_BASELINE_COMPARISON = BASELINE_MANIFEST_PATH is not None
else:
    ENABLE_BASELINE_COMPARISON = ENABLE_BASELINE_COMPARISON_RAW not in {'0', 'false', 'no'}
display(pd.DataFrame([{
    'gpu_name': GPU_NAME,
    'gpu_safe_profile': GPU_SAFE_PROFILE.name,
    'precision_dtype': PRECISION_CONFIG['dtype_name'],
}]))

print('KAGGLE_JSON_DRIVE_PATH =', KAGGLE_JSON_DRIVE_PATH)
print('BASELINE_MANIFEST_PATH =', BASELINE_MANIFEST_PATH)
print('ENABLE_BASELINE_COMPARISON =', ENABLE_BASELINE_COMPARISON)

config = NewsSummaryFineTuneConfig(
    base_model_name=BASE_MODEL_NAME,
    dataset_slug=DATASET_SLUG,
    output_root=OUTPUT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    cache_dir=CACHE_DIR,
    kaggle_json_drive_path=KAGGLE_JSON_DRIVE_PATH,
    csv_filename=CSV_FILENAME,
    article_column=ARTICLE_COLUMN,
    summary_column=SUMMARY_COLUMN,
    aux_headline_column=AUX_HEADLINE_COLUMN,
    split_column=SPLIT_COLUMN,
    target_split=TARGET_SPLIT,
    baseline_manifest_path=BASELINE_MANIFEST_PATH,
    random_seed=RANDOM_SEED,
    max_eval_samples=MAX_EVAL_SAMPLES,
    max_seq_length=MAX_SEQ_LENGTH,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_steps=WARMUP_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    weight_decay=WEIGHT_DECAY,
)
baseline_manifest = load_baseline_manifest(BASELINE_MANIFEST_PATH)
if ENABLE_BASELINE_COMPARISON and BASELINE_MANIFEST_PATH is None:
    print('Baseline comparison requires NEWS_SUMMARY_BASELINE_MANIFEST_PATH from notebooks/model_eval_news_summary_colab.ipynb')
elif not ENABLE_BASELINE_COMPARISON:
    print('Baseline comparison disabled. Set NEWS_SUMMARY_BASELINE_MANIFEST_PATH or NEWS_SUMMARY_ENABLE_BASELINE_COMPARISON=1 to enable it.')
require_baseline_manifest(baseline_manifest, ENABLE_BASELINE_COMPARISON)
prompt_profile = build_production_adapted_prompt_profile()
run_paths = build_run_paths(CHECKPOINT_ROOT, RUN_NAME)
for path in run_paths.values():
    if path.suffix:
        path.parent.mkdir(parents=True, exist_ok=True)
    else:
        path.mkdir(parents=True, exist_ok=True)

display(Markdown('## Prompt adaptation note'))
display(Markdown(prompt_profile.adaptation_note))
display(pd.DataFrame([config.to_serializable_dict()]).T.rename(columns={0: 'value'}))


In [ ]:
import json
import getpass
import shutil

def ensure_notebook_kaggle_credentials(kaggle_json_drive_path: Path) -> str:
    username = os.environ.get('KAGGLE_USERNAME', '').strip()
    key = os.environ.get('KAGGLE_KEY', '').strip()
    home_kaggle_dir = Path.home() / '.kaggle'
    home_kaggle_dir.mkdir(parents=True, exist_ok=True)
    home_kaggle_path = home_kaggle_dir / 'kaggle.json'

    print('Credential check:')
    print('  KAGGLE_JSON_DRIVE_PATH =', kaggle_json_drive_path)
    print('  kaggle.json exists in Drive =', kaggle_json_drive_path.exists())
    print('  env credentials set =', bool(username and key))

    if kaggle_json_drive_path.exists():
        home_kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(kaggle_json_drive_path, home_kaggle_path)
        try:
            home_kaggle_path.chmod(0o600)
        except Exception:
            pass
        os.environ['KAGGLE_CONFIG_DIR'] = str(home_kaggle_dir)
        print('Using kaggle.json from Drive')
        return str(kaggle_json_drive_path)

    if username and key:
        kaggle_json_drive_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {'username': username, 'key': key}
        kaggle_json_drive_path.write_text(json.dumps(payload), encoding='utf-8')
        try:
            kaggle_json_drive_path.chmod(0o600)
        except Exception:
            pass
        home_kaggle_path.write_text(json.dumps(payload), encoding='utf-8')
        try:
            home_kaggle_path.chmod(0o600)
        except Exception:
            pass
        os.environ['KAGGLE_CONFIG_DIR'] = str(home_kaggle_dir)
        print('Saved kaggle.json from environment variables')
        return str(kaggle_json_drive_path)

    if not IN_COLAB:
        raise FileNotFoundError(
            'Kaggle credential not found. Set KAGGLE_USERNAME/KAGGLE_KEY or place kaggle.json at '
            f'{kaggle_json_drive_path}.'
        )

    print('Kaggle credential not found. Enter credentials once to save them in Drive for future runs.')
    username = input('KAGGLE_USERNAME: ').strip()
    key = getpass.getpass('KAGGLE_KEY: ').strip()
    if not username or not key:
        raise FileNotFoundError(
            'Kaggle credential input was empty. Set KAGGLE_USERNAME/KAGGLE_KEY or place kaggle.json at '
            f'{kaggle_json_drive_path}.'
        )

    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = key
    payload = {'username': username, 'key': key}
    kaggle_json_drive_path.parent.mkdir(parents=True, exist_ok=True)
    kaggle_json_drive_path.write_text(json.dumps(payload), encoding='utf-8')
    home_kaggle_path.write_text(json.dumps(payload), encoding='utf-8')
    for path in (kaggle_json_drive_path, home_kaggle_path):
        try:
            path.chmod(0o600)
        except Exception:
            pass
    os.environ['KAGGLE_CONFIG_DIR'] = str(home_kaggle_dir)
    print(f'Saved kaggle.json to {kaggle_json_drive_path}')
    return str(kaggle_json_drive_path)

credential_source = ensure_notebook_kaggle_credentials(KAGGLE_JSON_DRIVE_PATH)
print('Kaggle credential ready from', credential_source)


In [ ]:
dataset_dir = download_dataset_if_needed(DATASET_SLUG, CACHE_DIR, force_redownload=FORCE_REDOWNLOAD)
csv_path = resolve_csv_file(dataset_dir, CSV_FILENAME)
raw_df = read_csv_robust(csv_path)
resolved_split_column, resolved_target_split = resolve_training_split_config(config, baseline_manifest, raw_df)
validate_baseline_protocol_compatibility(config, baseline_manifest, csv_path.name, resolved_split_column, resolved_target_split)
clean_df = load_training_dataframe(
    csv_path=csv_path,
    article_column=ARTICLE_COLUMN,
    summary_column=SUMMARY_COLUMN,
    aux_headline_column=AUX_HEADLINE_COLUMN,
    split_column=resolved_split_column,
    target_split=resolved_target_split,
)
frozen_eval_ids_path = resolve_frozen_eval_ids_path(config, baseline_manifest)
train_df, eval_df, split_profile = split_train_eval_by_frozen_ids(
    clean_df=clean_df,
    frozen_eval_ids_path=frozen_eval_ids_path,
    max_eval_samples=MAX_EVAL_SAMPLES,
    random_seed=RANDOM_SEED,
)
print('csv_path =', csv_path)
print('resolved_split_column =', resolved_split_column, 'resolved_target_split =', resolved_target_split)
print('train_rows =', len(train_df), 'eval_rows =', len(eval_df))
display(train_df.head(2))
display(eval_df.head(2))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=getattr(torch, PRECISION_CONFIG['dtype_name']),
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=RANDOM_SEED,
)

train_records = prepare_training_records(train_df, tokenizer)
eval_records = prepare_training_records(eval_df, tokenizer)
train_dataset = Dataset.from_pandas(train_records[['text']], preserve_index=False)
eval_dataset = Dataset.from_pandas(eval_records[['text']], preserve_index=False)
resume_from_checkpoint = resolve_resume_checkpoint(run_paths['run_root'], run_paths['archive_dir'])
print('resume_from_checkpoint =', resume_from_checkpoint)


In [ ]:
training_args = SFTConfig(**build_sft_config_kwargs(
    SFTConfig,
    output_dir=str(run_paths['archive_dir']),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    save_total_limit=max(SAVE_TOTAL_LIMIT, 2),
    report_to='none',
    seed=RANDOM_SEED,
    max_seq_length=MAX_SEQ_LENGTH,
    bf16=PRECISION_CONFIG['bf16'],
    fp16=PRECISION_CONFIG['fp16'],
    eval_accumulation_steps=1,
))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    dataset_text_field='text',
    callbacks=[
        build_checkpoint_callback(run_paths['run_root'], run_paths['archive_dir'], keep_last=SAVE_TOTAL_LIMIT),
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
    ],
)

training_manifest = build_training_manifest(
    config=config,
    run_name=RUN_NAME,
    run_paths=run_paths,
    baseline_manifest=baseline_manifest,
    split_profile=split_profile,
    resume_from_checkpoint=resume_from_checkpoint,
)
write_training_manifest(run_paths['training_manifest_path'], training_manifest)

train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
checkpoint_index = refresh_checkpoint_index(
    run_root=run_paths['run_root'],
    archive_dir=run_paths['archive_dir'],
    keep_last=SAVE_TOTAL_LIMIT,
    best_checkpoint_path=trainer.state.best_model_checkpoint,
)
trainer.save_state()
model.save_pretrained(run_paths['adapter_dir'])
tokenizer.save_pretrained(run_paths['adapter_dir'])

training_manifest['trainer_state'] = {
    'best_model_checkpoint': trainer.state.best_model_checkpoint,
    'global_step': int(trainer.state.global_step),
    'epoch': float(trainer.state.epoch or 0.0),
}
training_manifest['checkpoint_index'] = checkpoint_index
write_training_manifest(run_paths['training_manifest_path'], training_manifest)
display(Markdown(f"- Best checkpoint: `{trainer.state.best_model_checkpoint}`"))
display(Markdown(f"- Adapter dir: `{run_paths['adapter_dir']}`"))


In [ ]:
post_train_eval_config = build_post_train_baseline_config(
    baseline_manifest=baseline_manifest,
    model_path=run_paths['adapter_dir'],
    results_dir=run_paths['post_eval_dir'],
    frozen_eval_ids_path=frozen_eval_ids_path,
    kaggle_json_drive_path=KAGGLE_JSON_DRIVE_PATH,
)
import reasoning_nlp.eval.news_summary_baseline as news_summary_baseline_module
importlib.reload(news_summary_baseline_module)
print('Running post-train eval from', inspect.getsourcefile(news_summary_baseline_module))

post_train_eval = news_summary_baseline_module.run_baseline_evaluation(
    config=post_train_eval_config,
    prompt_profile=prompt_profile,
    force_redownload=False,
)
display(pd.DataFrame([post_train_eval['metrics']]).T.rename(columns={0: 'value'}))
if ENABLE_BASELINE_COMPARISON:
    comparison_df = build_before_after_comparison(
        baseline_metrics=baseline_manifest.get('metrics', {}),
        finetuned_metrics=post_train_eval['metrics'],
    )
    comparison_df.to_csv(run_paths['before_after_csv'], index=False)
    display(comparison_df)
display(Markdown((Path(post_train_eval['run_dir']) / 'baseline_report.md').read_text(encoding='utf-8')))
